# Knee v6 round 1: token head and canonical geometry (fold 0)
Question: can the model beat the fold-0 baseline (macro ~0.7795, KEY-4 ~0.7547; seed noise ~0.005 KEY-4) with
(a) `--head tokens`: per-label attention over every 2.5D window instead of one pooled token per series, and
(b) canonical orientation + right-knee mirroring (`--geom_csv`), computed from DICOM headers, no re-caching?
Same backbone (EfficientNet-B0, 160 px), same recipe and labels as the baseline, so inference stays as light as now.
Inputs: competition data, `knee-mri-pipeline` dataset holding **knee_pipeline_v6**, caching output (cache + labels_soft.csv),
training output (oof_all.csv). GPU T4 x2, internet ON, Save & Run All. Cell 2 (audit) decides whether the geometry arms run.

In [1]:
import os, sys, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    print(f"{name:7s} {p}  exists={p.exists()}")
    assert p.exists(), f"{name} not found: fix the path above (or attach the input)"
for f in ("geom.py", "geom_table.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: the pipeline dataset does not hold knee_pipeline_v6 yet"
assert "--head" in (CODE / "train.py").read_text() and "--geom_csv" in (CODE / "train.py").read_text(), "train.py is not v6"
print("cache files:", len(list(CACHE.glob("*.npy"))), "| labels rows:", len(pd.read_csv(LABELS)), "| baseline rows:", len(pd.read_csv(BASE)))

CODE    /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline  exists=True
DATA    /kaggle/input/competitions/rsna-knee-abnormality-detection  exists=True
CACHE   /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache  exists=True
LABELS  /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv  exists=True
BASE    /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv  exists=True
cache files: 4407 | labels rows: 4407 | baseline rows: 4407


In [2]:
# ---- 1. unit tests + smoke test (seconds). Anything failing here would waste a 65-minute job. ----
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-1500:]}{r.stderr[-1500:]}")
    assert r.returncode == 0, f"{t} failed: stop and send me this output"

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS
/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/tests/../model.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fuse = nn.TransformerEncoder(layer, n_layers)



In [3]:
# ---- 2. geometry audit on the TRAIN set (CPU, headers only) ----
t0 = time.time()
r = subprocess.run([sys.executable, str(CODE / "geom_table.py"), "--split", "train", "--data_dir", str(DATA),
                    "--out", str(W / "geom.csv"), "--K", "1", "--workers", "16"], capture_output=True, text=True, cwd=str(CODE))
print(r.stdout[-3000:], r.stderr[-1500:], f"\n[{time.time()-t0:.0f}s]")
assert r.returncode == 0 and (W / "geom.csv").exists(), "audit failed: send me the output above"
g = pd.read_csv(W / "geom.csv", dtype={"StudyInstanceUID": str}).fillna({"side": ""})
cs = [c for c in g.columns if c.startswith("code_")]
changed = float(((g[cs] > 0).any(axis=1) | (g.side == "R")).mean())
RUN_GEO = changed >= 0.05
print(f"\nstudies changed by the geometry fix: {changed:.1%}  ->  geometry arms {'WILL' if RUN_GEO else 'will NOT'} run")


===== GEOMETRY AUDIT (4407 studies, train) -> /kaggle/working/geom.csv
Sagittal  untouched(-1)   0.0% | already canonical   0.0% | transpose   0.0% | flip cols   0.0% | flip rows   0.0% | reverse slices 100.0%
Coronal   untouched(-1)   0.0% | already canonical 100.0% | transpose   0.0% | flip cols   0.0% | flip rows   0.0% | reverse slices   0.0%
Axial     untouched(-1)   0.0% | already canonical   0.0% | transpose   0.0% | flip cols   0.0% | flip rows   0.0% | reverse slices 100.0%

laterality source: {'tag': 2196, 'geometry': 2136, 'unknown': 75}
side: L 46.2% | R 52.1% | unknown 1.7%
tag vs geometry disagree on 26 of 2196 tagged studies (1.2%)
studies where the fix changes at least one array: 100.0%
  
[342s]

studies changed by the geometry fix: 100.0%  ->  geometry arms WILL run


In [4]:
# ---- 3. launch: two GPUs, each runs its queue sequentially. ~65 min per job. ----
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--only_folds", "0", "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
GEO = ["--geom_csv", str(W / "geom.csv")]
TOK = ["--head", "tokens"]
if RUN_GEO:
    QUEUES = {0: [("v6_tok", TOK), ("v6_both", TOK + GEO)],
              1: [("v6_geo", GEO), ("v6_both_s7", TOK + GEO + ["--seed", "7"])]}
else:
    QUEUES = {0: [("v6_tok", TOK)], 1: [("v6_tok_s7", TOK + ["--seed", "7"])]}
LOGS = {}

def run_queue(gpu, jobs):
    for name, extra in jobs:
        out = W / name
        if (out / "oof.csv").exists():
            print(f"[{name}] oof.csv exists, skipped"); continue
        cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + extra + ["--out", str(out)]
        log = W / f"{name}.log"; LOGS[name] = log
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
        with open(log, "w") as fh:
            p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env, cwd=str(CODE))
            rc = p.wait()
        print(f"[{name}] exit code {rc}", flush=True)

threads = [threading.Thread(target=run_queue, args=(gpu, jobs)) for gpu, jobs in QUEUES.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(900)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if l.strip()] if f.exists() else []
        ep = [l for l in ls if "ep " in l]
        last[n] = (ep[-1] if ep else (ls[-1][:120] if ls else "(none yet)"))
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished")

[v6_tok] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --out /kaggle/working/v6_tok
[v6_geo] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --geom_csv /kaggle/working/geom.csv --out /kaggle/working/v6_geo
[0.25 h] {
 "v6_tok": "[f0] ep 2/12 loss 0.7814 | val macro-AUC 0.7243 | gold 0.6550 | 414s",
 "v6_geo": "[f0] ep 2/12 loss 0.7835 | val macro-AUC 0.7126 | gold 0.6402 | 419s

In [5]:
# ---- 4. analysis: fold-0 AUC on confident labels (y>=0.9 or <=0.1), old labels, vs baseline and seed noise ----
from sklearn.metrics import roc_auc_score
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
arms = {n: pd.read_csv(W / n / "oof.csv") for q in QUEUES.values() for n, _ in q if (W / n / "oof.csv").exists()}
assert arms, "no arm finished"
first = next(iter(arms.values()))
val = first[[f"p_{l}" for l in LAB]].notna().all(axis=1).values            # fold-0 rows
assert val.sum() > 500, f"fold-0 rows: {val.sum()}"
assert np.allclose(base.loc[val, [f"y_{l}" for l in LAB]].values, first.loc[val, [f"y_{l}" for l in LAB]].values, atol=1e-6), \
       "fixed labels differ between baseline and arm: the comparison would be invalid"

def auc(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len(set((y[m] >= 0.9))) == 2 else np.nan

def table(preds):
    return pd.DataFrame({n: {l: auc(base.loc[val, f"y_{l}"].values, p[l]) for l in LAB} for n, p in preds.items()})

P = {"baseline": {l: base.loc[val, f"p_{l}"].values for l in LAB}}
for n, d in arms.items(): P[n] = {l: d.loc[val, f"p_{l}"].values for l in LAB}
T = table(P); T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print(T.round(3).to_string())
print("\nreference fold-0 KEY-4 from earlier runs: baseline 0.755 | seed7 0.760 | seed11 0.759 (range 0.005)")
rng = np.random.default_rng(0); idx = np.flatnonzero(val)
for n in arms:
    d = []
    for _ in range(300):
        b = rng.choice(len(idx), len(idx))
        yb = lambda l: base.loc[idx[b], f"y_{l}"].values
        ka = np.nanmean([auc(yb(l), arms[n].loc[idx[b], f"p_{l}"].values) for l in KEY4])
        kb = np.nanmean([auc(yb(l), base.loc[idx[b], f"p_{l}"].values) for l in KEY4])
        d.append(ka - kb)
    delta = T.loc["KEY-4", n] - T.loc["KEY-4", "baseline"]
    verdict = "WIN (> 2x seed range)" if delta > 0.01 else ("harm" if delta < -0.01 else "tie")
    print(f"{n:12s} KEY-4 delta {delta:+.4f}  90% CI [{np.percentile(d,5):+.3f}, {np.percentile(d,95):+.3f}] (study resampling only) -> {verdict}")
print("\nFold 0 is the EASIEST fold: a fold-0 gain still needs a 5-fold run (and the public LB) before you trust it.")

                  baseline  v6_tok  v6_both  v6_geo  v6_both_s7
ACL                  0.833   0.856    0.858   0.828       0.865
MCL                  0.713   0.737    0.732   0.703       0.739
Medial Meniscus      0.757   0.764    0.757   0.765       0.785
Lateral Meniscus     0.715   0.717    0.716   0.715       0.711
Medial OA            0.861   0.861    0.864   0.856       0.873
Lateral OA           0.749   0.744    0.743   0.727       0.748
PF OA                0.722   0.745    0.741   0.711       0.734
Effusion             0.801   0.804    0.814   0.800       0.819
Synovitis            0.799   0.771    0.776   0.781       0.775
Baker's              0.747   0.758    0.768   0.804       0.837
Contusion            0.808   0.797    0.822   0.800       0.843
Fracture             0.848   0.876    0.883   0.845       0.880
MACRO                0.779   0.786    0.790   0.778       0.801
KEY-4                0.755   0.768    0.766   0.753       0.775

reference fold-0 KEY-4 from earlier run